# Title
TabPFN Benchmark
## Purpose
Explain how the TabPFN challenger benchmark is wired into the package helpers without forcing a training run.
## Inputs
`data/gold/listings_gold.xlsx`, the `run_tabpfn_regression` helper, and the optional TabPFN dependency.
## Outputs
TabPFN predictions and benchmark rows under `results/benchmarks/...` when training is enabled.
## Key Takeaways
This notebook is safe to open on a minimal environment because execution is opt-in and depends on the advanced stack.

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split

def resolve_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / 'config.py').exists():
            return candidate
    raise FileNotFoundError('Could not find config.py in the current path ancestry')

PROJECT_ROOT = resolve_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from config import LISTINGS_GOLD, RESULTS_DIR, TARGET_VARIABLE, NUMERIC_FEATURES, CATEGORICAL_FEATURES
from elferspot_listings.modeling.challengers import run_tabpfn_regression
from elferspot_listings.modeling.train import train_baseline_models

BENCHMARK_DIR = RESULTS_DIR / 'benchmarks' / '07_tabpfn_benchmark'
RUN_TRAINING = False
RUN_TABPFN = True
TABPFN_MODEL_PATH = 'default'
TABPFN_DEVICE = 'cpu'
TABPFN_RANDOM_STATE = 42
ADVANCED_DEPENDENCIES_NOTE = 'Install the optional stack with `python -m pip install -e ".[advanced]"`.'

In [ ]:
gold_exists = LISTINGS_GOLD.exists()
metrics_path = BENCHMARK_DIR / 'metrics.json'
predictions_path = BENCHMARK_DIR / 'predictions.csv'

print(f'Gold input: {LISTINGS_GOLD}')
print(f'Gold present: {gold_exists}')
print(f'Benchmark directory: {BENCHMARK_DIR}')
print('TabPFN helper imports are ready for opt-in training or artifact loading.')
print(ADVANCED_DEPENDENCIES_NOTE)

if gold_exists:
    gold_preview = pd.read_excel(LISTINGS_GOLD, nrows=3)
    print('\nGold preview:')
    print(gold_preview.to_string(index=False))

In [ ]:
if RUN_TRAINING and gold_exists:
    gold_df = pd.read_excel(LISTINGS_GOLD)
    if TARGET_VARIABLE not in gold_df.columns:
        raise KeyError(f'Target column {TARGET_VARIABLE!r} missing from gold dataset.')
    feature_cols = [c for c in NUMERIC_FEATURES + CATEGORICAL_FEATURES if c in gold_df.columns]
    X = gold_df[feature_cols]
    y = gold_df[TARGET_VARIABLE]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=TABPFN_RANDOM_STATE
    )
    model, predictions, metadata = run_tabpfn_regression(
        X_train,
        y_train,
        X_test,
        random_state=TABPFN_RANDOM_STATE,
        model_path=TABPFN_MODEL_PATH,
        device=TABPFN_DEVICE,
    )
    print(f'TabPFN fitted in {metadata["runtime_seconds"]:.1f}s ({metadata["notes"]}).')
    print('TabPFN is enabled through run_tabpfn_regression(...) or train_baseline_models(..., run_tabpfn=True).')
elif metrics_path.exists():
    metrics = json.loads(metrics_path.read_text(encoding='utf-8'))
    print('Loaded existing benchmark metrics:')
    print(pd.DataFrame(metrics).T.sort_values('mae_eur').to_string())
    if predictions_path.exists():
        predictions = pd.read_csv(predictions_path)
        tabpfn_rows = predictions[predictions['model_name'].str.startswith('tabpfn')]
        print('\nTabPFN prediction sample:')
        print(tabpfn_rows.head().to_string(index=False))
else:
    print('No TabPFN outputs found yet. Keep RUN_TRAINING=False for review-only use, or enable it when advanced dependencies are installed.')